In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import os, argparse
from scipy import misc
import cv2

from lib.networks import PVT_CASCADE
from utils.dataloader import test_dataset
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

/nfs/P111yhchen/anaconda3/envs/medseg/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/P111yhchen/code/medseg/CASCADE/lib/pvtv2.py:388: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  class pvt_v2_b0(PyramidVisionTransformerImpr):
/home/P111yhchen/code/medseg/CASCADE/lib/pvtv2.py:398: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  class pvt_v2_b1(PyramidVisionTransformerImpr):
/home/P111yhchen/code/medseg/CASCADE/lib/pvtv2.py:406: UserWarning: Overwriting pvt_v2_b2 in registry with lib.pvtv2.pvt_v2_b2. This is because the n

In [2]:
testsize = 352
device_id = 1
device = torch.device(f'cuda:{device_id}')
torch.cuda.set_device(device_id)
pth_path = '/nfs/P111yhchen/code/medseg/CASCADE/models/polyp/ffd/best.pth'

model = PVT_CASCADE(n_class=1).to(device)
model.eval()
model.load_state_dict(torch.load(pth_path, map_location=device))

pvt_v2_b2


/home/P111yhchen/code/medseg/CASCADE/lib/networks.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  save_model = torch.load(path)


Model decoder:  created, param count: 339109


/tmp/ipykernel_222568/2514500420.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(pth_path, map_location=device))


<All keys matched successfully>

In [3]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).cuda()
STD = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).cuda()

In [4]:
stds = []
preds = []
#datasets = [ 'CVC-ClinicDB', 'Kvasir', 'CVC-300', 'CVC-ColonDB', 'ETIS-LaribPolypDB']
datasets = ['ETIS-LaribPolypDB']
for _data_name in datasets:
#for _data_name in ['TrainDataset']:
    ##### put data_path here #####
    data_path = '/home/P111yhchen/data/polyp/TestDataset/{}'.format(_data_name)
    #data_path = '/home/P111yhchen/data/polyp/{}'.format(_data_name)
    print('Evaluating ' + data_path)
    
    image_root = '{}/images/'.format(data_path)
    gt_root = '{}/masks/'.format(data_path)
    num1 = len(os.listdir(gt_root))
    test_loader = test_dataset(image_root, gt_root, 352)
    
    for i in range(num1):
        image, gt, name = test_loader.load_data()
        image = image.cuda()
        x1, x2, x3, x4, res4 = model(image) # forward
        preds.append(x4.view(x4.shape[1], -1).clone().detach())
samples = torch.cat(preds, dim=1)
std = torch.std(samples, dim=1)


Evaluating /home/P111yhchen/data/polyp/TestDataset/ETIS-LaribPolypDB


In [5]:
#for i in range(samples.shape[0]):
#    show_sample = samples[i].detach().cpu().numpy()
#    print(samples[i].min(), samples[i].max(), std[i])
#    bins = np.arange(-20.0, 20.0, 1)
#    plt.hist(show_sample, bins=bins, edgecolor='black')
#    
#    #plt.title('Standard Distribution - IAN')
#    plt.xlabel('std')
#    plt.ylabel('frequency')
#    plt.grid(True, linestyle='--', linewidth=0.5)
#    #plt.xticks(bins[::4])
#    #plt.yticks(np.arange(0, max(plt.ylim()) + 10, 10))
#    plt.show()
#show_stds = std.detach().cpu().numpy()
#bins = np.arange(0.0, 2.8, 0.1)
#print(std.min(), std.max(), std.shape)
#plt.hist(show_stds, bins=bins, edgecolor='black')
#
#plt.title('Standard Distribution - IAN')
#plt.xlabel('std')
#plt.ylabel('frequency')
#plt.grid(True, linestyle='--', linewidth=0.5)
#plt.xticks(bins[::4])
#plt.yticks(np.arange(0, 30, 5))
#plt.show()

In [6]:
value, indice = torch.topk(std, 10)
print(value)

tensor([1.7349, 1.4241, 1.3659, 1.3467, 1.3240, 1.2893, 1.2767, 1.2746, 1.2565,
        1.2546], device='cuda:1')


In [7]:
show_samples = samples.detach().cpu().numpy()
#cov = np.cov(show_samples)
#d = show_samples.shape[0]
#epsilon = 1e-6
#cov += np.eye(cov.shape[0]) * epsilon  # Add a small regularization term
#eigenvalues, _ = np.linalg.eigh(cov)  # Eigenvalues of the covariance matrix
#log_det_cov = np.sum(np.log(eigenvalues))  # Sum of log eigenvalues
#entropy = 0.5 * (d * np.log(2 * np.pi * np.e) + log_det_cov)
#print(entropy)

In [8]:
#for j in range(5):
#    i = indice[j]
#    show_sample = samples[i].detach().cpu().numpy()
#    print(samples[i].min(), samples[i].max(), std[i])
#    bins = np.arange(-10.0, 10.0, 0.1)
#    plt.hist(show_sample, bins=bins, edgecolor='black')
#    
#    #plt.title('Standard Distribution - IAN')
#    plt.xlabel('std')
#    plt.ylabel('frequency')
#    plt.grid(True, linestyle='--', linewidth=0.5)
#    #plt.xticks(bins[::4])
#    #plt.yticks(np.arange(0, max(plt.ylim()) + 10, 10))
#    plt.show()

In [9]:
#def estimate_intrinsic_dim_pca(X, threshold=0.95):
#    pca = PCA()
#    pca.fit(X)
#    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
#    d_int = np.searchsorted(cumulative_variance, threshold) + 1
#    return d_int, cumulative_variance
#
## 假設 X 是 shape=(N, 64) 的 feature map
#d_int, cum_var = estimate_intrinsic_dim_pca(show_samples.T)
#print(f"Intrinsic dimension (PCA @95%): {d_int}")

In [10]:
from sklearn.neighbors import NearestNeighbors
def estimate_intrinsic_dim_twonn(X):
    N = X.shape[0]
    num_samples = N // 100
    random_indices = np.random.choice(N, size=num_samples, replace=False)
    sampled_X = X[random_indices]

    nbrs = NearestNeighbors(n_neighbors=3).fit(sampled_X)
    distances, _ = nbrs.kneighbors(X)
    
    r1 = distances[:, 1]+1e-8
    r2 = distances[:, 2]
    print(r1.min(), r2.min())
    mu = r2 / r1
    log_mu = np.log(mu)
    id_est = 1 / (np.mean(log_mu))
    return id_est

# 使用範例
id_est = estimate_intrinsic_dim_twonn(show_samples.T)
print(f"Estimated intrinsic dimension (Two-NN): {id_est:.2f}")

1e-08 0.1375632159718379
Estimated intrinsic dimension (Two-NN): 27.83


In [11]:
"""
#for _data_name in ['CVC-300', 'CVC-ClinicDB', 'Kvasir', 'CVC-ColonDB', 'ETIS-LaribPolypDB']:
for _data_name in ['CVC-300', 'CVC-ColonDB', 'ETIS-LaribPolypDB']:
#for _data_name in ['TrainDataset']:
    ##### put data_path here #####
    data_path = '/local_data/dataset/polyp/TestDataset/{}'.format(_data_name)
    #data_path = '/local_data/dataset/polyp/{}'.format(_data_name)
    print('Evaluating ' + data_path)
    
    image_root = '{}/images/'.format(data_path)
    gt_root = '{}/masks/'.format(data_path)
    num1 = len(os.listdir(gt_root))
    test_loader = test_dataset(image_root, gt_root, 352)
    
    for i in range(num1):
        image, gt, name = test_loader.load_data()
        gt = np.asarray(gt, np.float32)
        gt /= (gt.max() + 1e-8)
        image = image.cuda()
        img = torch.clamp(image * STD + MEAN, 0, 1)
        h,w = gt.shape
        gt_tensor = torch.tensor(gt).cuda().view(1,1,h,w)
        gt_tensor = F.upsample(gt_tensor, size=(352,352), mode='bilinear', align_corners=False)       
        
        x1, x2, x3, x4, res4 = model(image) # forward
        
        pca = PCA(n_components=3)
        #x1_ = x1[0].view(320, 22*22).permute(1,0).detach().cpu().numpy()
        #feature_map_pca = pca.fit_transform(x1_)  # Shape: (176*176, 3)
        #feature_map_pca -= feature_map_pca.min(axis=0)  # Shift to positive
        #feature_map_pca /= feature_map_pca.max(axis=0)  # Scale to [0, 1]
        #feature_map_rgb = feature_map_pca.reshape(22, 22, 3)
        
        #x2_ = x2[0].view(128, 44*44).permute(1,0).detach().cpu().numpy()
        #feature_map_pca2 = pca.fit_transform(x2_)  # Shape: (176*176, 3)
        #feature_map_pca2 -= feature_map_pca2.min(axis=0)  # Shift to positive
        #feature_map_pca2 /= feature_map_pca2.max(axis=0)  # Scale to [0, 1]
        #feature_map_rgb2 = feature_map_pca2.reshape(44, 44, 3)
        
        #x3_ = x3[0].view(64, 88*88).permute(1,0).detach().cpu().numpy()
        #feature_map_pca3 = pca.fit_transform(x3_)  # Shape: (176*176, 3)
        #feature_map_pca3 -= feature_map_pca3.min(axis=0)  # Shift to positive
        #feature_map_pca3 /= feature_map_pca3.max(axis=0)  # Scale to [0, 1]
        #feature_map_rgb3 = feature_map_pca3.reshape(88, 88, 3)

        x4_ = x4[0].view(64, 88*88).permute(1,0).detach().cpu().numpy()
        feature_map_pca4 = pca.fit_transform(x4_)  # Shape: (176*176, 3)
        feature_map_pca4 -= feature_map_pca4.min(axis=0)  # Shift to positive
        feature_map_pca4 /= feature_map_pca4.max(axis=0)  # Scale to [0, 1]
        feature_map_rgb4 = feature_map_pca4.reshape(88, 88, 3)

        
        img_show = img.squeeze().permute(1,2,0).detach().cpu().numpy()
        msk_show = gt_tensor.squeeze().detach().cpu().numpy()
        
        fig, ax = plt.subplots(1, 2)
        ax[0].imshow(img_show)
        ax[0].axis('off')
        ax[1].imshow(feature_map_rgb4)
        ax[1].axis('off')
        #ax[2].imshow(feature_map_rgb2)
        #ax[2].axis('off')
        #ax[3].imshow(feature_map_rgb3)
        #ax[3].axis('off')
        plt.show()
"""

"\n#for _data_name in ['CVC-300', 'CVC-ClinicDB', 'Kvasir', 'CVC-ColonDB', 'ETIS-LaribPolypDB']:\nfor _data_name in ['CVC-300', 'CVC-ColonDB', 'ETIS-LaribPolypDB']:\n#for _data_name in ['TrainDataset']:\n    ##### put data_path here #####\n    data_path = '/local_data/dataset/polyp/TestDataset/{}'.format(_data_name)\n    #data_path = '/local_data/dataset/polyp/{}'.format(_data_name)\n    print('Evaluating ' + data_path)\n    \n    image_root = '{}/images/'.format(data_path)\n    gt_root = '{}/masks/'.format(data_path)\n    num1 = len(os.listdir(gt_root))\n    test_loader = test_dataset(image_root, gt_root, 352)\n    \n    for i in range(num1):\n        image, gt, name = test_loader.load_data()\n        gt = np.asarray(gt, np.float32)\n        gt /= (gt.max() + 1e-8)\n        image = image.cuda()\n        img = torch.clamp(image * STD + MEAN, 0, 1)\n        h,w = gt.shape\n        gt_tensor = torch.tensor(gt).cuda().view(1,1,h,w)\n        gt_tensor = F.upsample(gt_tensor, size=(352,352)